In [ ]:
from modules.dataset.make_dataset import make_dataset, dataset_split_by_recording, stratified_recording_folds, NpzDataPipeline, AugmentedSubset
from modules.nn import CoralOrdinal, DefaultCNN, coral_pos_weights, coral_targets, ordinal_oof_metrics, ordinal_predict
import polars as pl
import numpy as np
import torch
from torchinfo import summary
from torch.utils.data import Subset
import torch.nn as nn
import torch.optim as optim
from typing import Sequence
from sklearn.metrics import f1_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

In [ ]:
dataset: NpzDataPipeline = NpzDataPipeline("final_processed_dataset.npz")
device: torch.types.Device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [3]:
import os

# ---- Crash-proof run state: per-epoch autosave + resume (train_3class) ----
RUN_STATE_DIR: str = "run_state_3class"
os.makedirs(RUN_STATE_DIR, exist_ok=True)

def _ckpt_path(name: str) -> str:
    return os.path.join(RUN_STATE_DIR, f"{name}_latest.pth.tar")

def _atomic_torch_save(obj: dict, path: str) -> None:
    tmp = path + ".tmp"
    torch.save(obj, tmp)
    os.replace(tmp, path)  # crash mid-write never corrupts the good file

def resume_or_fresh(
    name: str,
    model: nn.Module,
    optimizer: optim.Optimizer,
    scheduler: optim.lr_scheduler.LRScheduler,
    scaler: torch.amp.GradScaler,
    num_epochs: int,
) -> tuple[int, dict[str, list[float]]]:
    """Load `run_state/{name}_latest.pth.tar` into model/opt/sched/scaler if it exists.
    Returns (epochs_done, history). If the run is already complete, also registers
    the model in trained_models so post-training eval works on a fresh kernel."""
    path = _ckpt_path(name)
    fresh: dict[str, list[float]] = {"train_loss": [], "train_acc": [], "train_F1": [], "val_loss": [], "val_acc": [], "val_F1": [], "val_auc": [], "val_pr_auc": []}
    if not os.path.exists(path):
        return 0, fresh
    ckpt = torch.load(path, weights_only=True, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    scaler.load_state_dict(ckpt["scaler_state_dict"])
    epochs_done: int = ckpt["epoch"] + 1  # 'epoch' is the 0-based index of the last COMPLETED epoch
    history: dict[str, list[float]] = ckpt["history"]
    for k in ("train_acc", "train_F1", "val_pr_auc"):  # old checkpoints predate these keys
        history.setdefault(k, []).extend([float("nan")] * (len(history["val_loss"]) - len(history.get(k, []))))
    if epochs_done >= num_epochs:
        trained_models[name] = model  # finished run: restore into eval registry, no retrain
    print(f"=> [{name}] resumed at epoch {epochs_done + 1}/{num_epochs}" if epochs_done < num_epochs
          else f"=> [{name}] already complete ({num_epochs}/{num_epochs} epochs) — restored, no retrain")
    return epochs_done, history

In [ ]:
# ---- Experiment setup (shared) ----
from modules.dataset.make_dataset import AugmentedSubset, aug_probs_from_counts
from modules.preprocessing.audio_process import WaveformAugmentor, SpecAugmentations, MelSpectrogramPipeline, SpectMFCCPipeline
from tqdm.auto import tqdm

num_epochs: int = 60  # epoch cap for ALL runs; early stop (15/3) lives in train_ordinal
N_FOLDS: int = 5
SEED: int = 42  # identical split across all experiments
BATCH_SIZE: int = 8

# n_mel=64 per Chang et al. (research_results.md fix #6): 64 mel bins @ 16 kHz
mel_train_plain: SpectMFCCPipeline = SpectMFCCPipeline(combind=True).to(device)  # no spec aug
mel_eval: SpectMFCCPipeline = SpectMFCCPipeline(combind=True).to(device)         # plain for validation

# speed: compile only the eval mel path; train paths stay eager (spec-aug masks/augmentor randomize per step)
mel_eval = torch.compile(mel_eval) if device.type == "cuda" else mel_eval  # CPU inductor needs MSVC cl.exe; skip
torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")


In [5]:
# ---- Recording-wise stratified split (shared across experiments) ----
# split by SOURCE RECORDING so sibling segments of one clip can't leak across
# splits; per-class recording allocation keeps each split's class mix ~ population
# instead of drifting monotonically train -> val -> test

dataset.load_data()
dataset.load_paths()

train_subset, val_subset, test_subset = dataset_split_by_recording(dataset, dataset.paths, 0.8, 0.1, seed=SEED)

# sanity: class mix per split (should now track the population mix closely)
LABELS_INV = {0: "healthy", 1: "COVID-19", 2: "symptomatic"}
for name, s in [("train", train_subset), ("val", val_subset), ("test", test_subset)]:
    counts = np.bincount([int(dataset.labels[i]) for i in s.indices], minlength=3)
    print(name, {LABELS_INV[k]: int(v) for k, v in enumerate(counts)})

healthy: 241 (21.4%)
COVID-19: 559 (49.6%)
symptomatic: 328 (29.1%)
total: 1128
train {'healthy': 170, 'COVID-19': 454, 'symptomatic': 279}
val {'healthy': 34, 'COVID-19': 53, 'symptomatic': 27}
test {'healthy': 37, 'COVID-19': 52, 'symptomatic': 22}


In [6]:
# ---- Per-class loss weights (inverse frequency, train split only) ----
train_labels = torch.tensor([int(dataset.labels[i]) for i in train_subset.indices])
counts = torch.bincount(train_labels, minlength=3).float()
w = 1.0 / counts.clamp(min=1)
w = w / w.sum() * 3.0  # normalize so mean weight is 1
print({LABELS_INV[i]: round(float(x), 4) for i, x in enumerate(w)})

{'healthy': 1.5123, 'COVID-19': 0.5663, 'symptomatic': 0.9215}


In [ ]:
_model = CoralOrdinal(DefaultCNN(num_classes=1))
# combined pipeline: 64 mel + 40 MFCC = 104 freq bins, 3 channels (spec, mfcc, delta)
# CORAL head: backbone -> shared scalar score -> (B, 2) cumulative logits (P(rank>low), P(rank>med))
summary(_model, input_size=(128, 3, 104, 32))

In [10]:
# ---- Plain 3-class training (healthy / COVID-19 / symptomatic) ----
def train_3class(name: str, use_wave: bool, use_spec: bool) -> dict[str, list[float]]:
    """Straight 3-class classification: raw labels 0/1/2 used directly, weighted CE.
    Crash-proof: state autosaves to run_state/{name}_latest.pth.tar every epoch;
    re-running resumes mid-run (or short-circuits if already complete)."""
    history: dict[str, list[float]] = {"train_loss": [], "train_acc": [], "train_F1": [], "val_loss": [], "val_acc": [], "val_F1": [], "val_auc": [], "val_pr_auc": []}

    
    # Rarity-scaled augment probs from train-split class counts: rarer classes
    # get more augmentation (most-common -> 0.5, rarest -> 0.95).
    counts3 = torch.bincount(train_labels, minlength=3).float()
    aug_probs: dict[int, float] = aug_probs_from_counts({k: float(counts3[k]) for k in range(3)})
    print(f"[{name}] augment_probs: " + ", ".join(f"{k}:{v:.2f}" for k, v in sorted(aug_probs.items())))

    criterion: nn.Module = FocalLoss(w.to(device))

    if device.type == "cuda":
        model: nn.Module = torch.compile(DefaultCNN(num_classes=3)).to(device)
    else:
        model = DefaultCNN(num_classes=3).to(device)  # speed: compile only on CUDA

    optimizer: optim.AdamW = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
    scaler: torch.amp.GradScaler = torch.amp.GradScaler(enabled=(device.type == "cuda"))
    scheduler: optim.lr_scheduler.CosineAnnealingLR = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=num_epochs, eta_min=1e-6
    )

    # crash recovery: pick up where the last run left off (0 + fresh history if none)
    epochs_done, history = resume_or_fresh(name, model, optimizer, scheduler, scaler, num_epochs)

    if use_spec:
        # per-run pipeline: SpecAugmentations holds this run's augment_probs
        mel_train: SpectMFCCPipeline = SpectMFCCPipeline(
            transforms=SpecAugmentations(augment_probs=aug_probs).to(device), combind=True
        ).to(device)
    else:
        mel_train = mel_train_plain

    # Train loader: AugmentedSubset (WaveformAugmentor) when use_wave, else plain Subset
    train_ds: AugmentedSubset | Subset = AugmentedSubset(
        dataset, train_subset.indices, WaveformAugmentor(16_000, augment_probs=aug_probs)
    ) if use_wave else Subset(dataset, train_subset.indices)

    train_loader: torch.utils.data.DataLoader = torch.utils.data.DataLoader(
        train_ds, BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True,
        persistent_workers=True, prefetch_factor=4,  # speed: no worker respawn / keep queues full
    )
    val_loader: torch.utils.data.DataLoader = torch.utils.data.DataLoader(
        Subset(dataset, val_subset.indices), BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True,
        persistent_workers=True, prefetch_factor=4,
    )

    for epoch in range(epochs_done, num_epochs):
        # Training Phase
        model.train()
        running_loss = 0.0
        train_preds: list[int] = []
        train_targets: list[int] = []

        for waveforms, labels in tqdm(train_loader, desc=f"[{name}] train {epoch+1}/{num_epochs}", leave=False):
            waveforms = waveforms.to(device, non_blocking=True)
            raw = labels.long().to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            # Features must run in fp32 OUTSIDE autocast: TimeStretch/complex spectrogram
            # ops produce NaN under fp16 autocast
            images = mel_train(waveforms, raw)  # (B, 3, F, T)
            if images.ndim == 3:  # pipeline dropped the channel dim on a 1-sample batch
                images = images.unsqueeze(0)

            with torch.amp.autocast(device_type=device.type, enabled=(device.type == "cuda")):
                outputs = model(images)
                loss = criterion(outputs, raw)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item()
            train_preds.extend(torch.argmax(outputs, dim=1).cpu().numpy())
            train_targets.extend(raw.cpu().numpy())

        scheduler.step()
        train_acc = 100.0 * float(np.mean(np.asarray(train_targets) == np.asarray(train_preds)))
        train_f1 = f1_score(train_targets, train_preds, average="macro")

        # Validation Phase
        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0
        all_preds: list[int] = []
        all_targets: list[int] = []
        all_probs: list[list[float]] = []

        with torch.inference_mode():
            for waveforms, labels in tqdm(val_loader, desc=f"[{name}] val {epoch+1}/{num_epochs}", leave=False):
                waveforms = waveforms.to(device, non_blocking=True)
                raw = labels.long().to(device, non_blocking=True)

                images = mel_eval(waveforms)  # no labels -> no spec aug; fp32 outside autocast
                if images.ndim == 3:
                    images = images.unsqueeze(0)

                with torch.amp.autocast(device_type=device.type, enabled=(device.type == "cuda")):
                    outputs = model(images)
                    loss = criterion(outputs, raw)

                all_preds.extend(torch.argmax(outputs, dim=1).cpu().numpy())
                all_targets.extend(raw.cpu().numpy())
                all_probs.extend(torch.softmax(outputs.float(), dim=1).cpu().numpy())

                val_loss += loss.item()
                total += raw.size(0)
                correct += (torch.argmax(outputs, dim=1) == raw).sum().item()

        val_macro_f1 = f1_score(all_targets, all_preds, average="macro")
        val_auc_score = roc_auc_score(all_targets, all_probs, multi_class="ovr", average="macro")
        val_pr_auc = average_precision_score(
            label_binarize(all_targets, classes=[0, 1, 2]), np.asarray(all_probs), average="macro")
        train_epoch_loss = running_loss / len(train_loader)
        val_epoch_loss = val_loss / len(val_loader)
        val_accuracy = 100.0 * correct / total

        history["train_loss"].append(train_epoch_loss)
        history["val_loss"].append(val_epoch_loss)
        history["val_F1"].append(val_macro_f1)
        history["val_acc"].append(val_accuracy)
        history["val_auc"].append(val_auc_score)
        history["train_acc"].append(train_acc)
        history["train_F1"].append(train_f1)
        history["val_pr_auc"].append(val_pr_auc)

        # autosave FULL state every epoch so a crash never loses progress
        _atomic_torch_save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict(),
            "history": history,
        }, _ckpt_path(name))
        if (epoch + 1) % 10 == 0 or epoch + 1 == num_epochs:
            print(f"=> [{name}] checkpoint saved (epoch {epoch+1}/{num_epochs})")

        print(f"[{name}] Epoch [{epoch+1}/{num_epochs}] -> "
              f"Train Loss: {train_epoch_loss:.4f} | Train Acc: {train_acc:.2f}% | Train F1: {train_f1:.4f} | "
              f"Val Loss: {val_epoch_loss:.4f} | "
              f"Val F1: {val_macro_f1:.4f} | Val AUC: {val_auc_score:.4f} | Val PR-AUC: {val_pr_auc:.4f} | "
              f"Val Acc: {val_accuracy:.2f}%")

    trained_models[name] = model  # kept for post-training eval
    return history

In [ ]:
# ---- Run: 3 augmentation configs (both / wave_only / spec_only) ----
import sys, atexit
CONFIGS: list[tuple[str, bool, bool]] = [("both", True, True), ("wave_only", True, False), ("spec_only", False, True)]
CSV_COLS: list[str] = ["train_loss", "train_acc", "train_F1", "val_loss", "val_acc", "val_F1", "val_auc", "val_pr_auc"]

results: dict[str, dict[str, list[float]]] = {}
trained_models: dict[str, nn.Module] = {}

os.makedirs("logs_3class", exist_ok=True)
os.makedirs("saved_models_3class", exist_ok=True)
LOG_PATH = r"logs_3class/three_class_experiment.txt"
_logf = open(LOG_PATH, "w", encoding="utf-8")  # fresh log per run; crash resume replays from ckpt anyway
sys.stdout = _logf
sys.stderr = _logf
atexit.register(_logf.close)

print("STARTING 3-CLASS EXPERIMENTS")
for cfg_name, uw, us in CONFIGS:
    key = f"three_class__{cfg_name}"
    print(f"\n--- {key} ---")
    hist = train_3class(key, use_wave=uw, use_spec=us)
    results[key] = hist
    pl.DataFrame({k: hist[k] for k in CSV_COLS}).write_csv(f"{key}_experiment_stats.csv")
    torch.save(trained_models[key].state_dict(), f"saved_models_3class/{key}.pth")

[three_class__both] train 1/60:   0%|          | 0/113 [01:06<?, ?it/s]

[three_class__both] val 1/60:   0%|          | 0/15 [00:58<?, ?it/s]

[three_class__both] train 2/60:   0%|          | 0/113 [00:00<?, ?it/s]

[three_class__both] val 2/60:   0%|          | 0/15 [00:00<?, ?it/s]

[three_class__both] train 3/60:   0%|          | 0/113 [00:00<?, ?it/s]

[three_class__both] val 3/60:   0%|          | 0/15 [00:00<?, ?it/s]

... (231 outputs left out)

[three_class__wave_only] val 59/60:   0%|          | 0/15 [00:00<?, ?it/s]

[three_class__wave_only] train 60/60:   0%|          | 0/113 [00:00<?, ?it/s]

[three_class__wave_only] val 60/60:   0%|          | 0/15 [00:00<?, ?it/s]

[three_class__spec_only] train 1/60:   0%|          | 0/113 [00:32<?, ?it/s]

In [ ]:
# ---- Test eval from checkpoints: per-class report + confusion matrices ----
os.makedirs("figures_3class", exist_ok=True)
CLASS_NAMES = [LABELS_INV[i] for i in range(3)]

for cfg_name, _, _ in CONFIGS:
    key = f"three_class__{cfg_name}"
    ckpt = torch.load(_ckpt_path(key), weights_only=True, map_location=device)
    m = DefaultCNN(num_classes=3).to(device)
    if any(k.startswith("_orig_mod.") for k in ckpt["model_state_dict"]):
        m = torch.compile(m)
    m.load_state_dict(ckpt["model_state_dict"])
    m.eval()

    loader = torch.utils.data.DataLoader(
        Subset(dataset, test_subset.indices), BATCH_SIZE, shuffle=False, num_workers=4,
    )
    preds: list[int] = []
    tgts: list[int] = []
    with torch.inference_mode():
        for waveforms, labels in loader:
            waveforms = waveforms.to(device, non_blocking=True)
            images = mel_eval(waveforms)
            if images.ndim == 3:
                images = images.unsqueeze(0)
            outputs = m(images)
            preds.extend(torch.argmax(outputs, dim=1).cpu().numpy())
            tgts.extend(labels.long().numpy())

    print(f"\n=== {key} (test) ===")
    print(classification_report(tgts, preds, target_names=CLASS_NAMES, digits=4))

    cm = confusion_matrix(tgts, preds)
    disp = ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES)
    disp.plot(cmap="viridis")
    plt.title(f"{key} - test confusion matrix")
    plt.tight_layout()
    plt.savefig(f"figures_3class/{key}_test_cm.png", dpi=200)
    plt.show()

In [ ]:
# ---- Summary table: max/min/final/avg/std per metric per experiment ----
def results_table(results: dict[str, dict[str, list[float]]]) -> pl.DataFrame:
    """Summary table per experiment: max / min / final / avg / std of each metric."""
    rows: list[dict] = []
    for name, hist in results.items():
        for metric in ("train_loss", "train_acc", "train_F1", "val_loss", "val_acc", "val_F1", "val_auc", "val_pr_auc"):
            vals: list[float] = hist[metric]
            rows.append({
                "experiment": name,
                "metric": metric,
                "max": round(max(vals), 4),
                "min": round(min(vals), 4),
                "final": round(vals[-1], 4),
                "avg": round(sum(vals) / len(vals), 4),
                "std": round(float(np.std(vals)), 4),
            })
    df = pl.DataFrame(rows)
    pl.Config.set_ascii_tables()  # cp1252 console can't render polars box chars
    with pl.Config(tbl_rows=-1, tbl_cols=-1, tbl_width_chars=120):
        print(df)
    return df

summary3 = results_table(results)
summary3.write_csv("three_class_experiment_summary.csv")

In [10]:
# ==== Multi-model training (DefaultCNN / BranchedCoughCNN / ResNet50 finetune), 3-class ====
from modules.nn import BranchedCoughCNN
from modules.nn.base_nn import ResNetModel

# pick model + augmentation BY STRING before running:
# model: "default" | "branched" | "resnet"   aug: "both" | "wave_only" | "spec_only"
TRAINED_CONFIGS: dict[str, list[str]] = {
    "default":  ["both", "wave_only", "spec_only"],
    "branched": ["both", "wave_only", "spec_only"],
    "resnet":   ["both", "wave_only", "spec_only"],
}
AUG_MAP: dict[str, tuple[bool, bool]] = {"both": (True, True), "wave_only": (True, False), "spec_only": (False, True)}
MODEL_PATIENCE: int = 20  # early stop: val_loss plateau of this many epochs
TRAINED_EPOCHS: int = 256
TRAINED_RUNS: list[tuple[str, str]] = [("default", "both"), ("branched", "both"), ("resnet", "both")]
print("Runs scheduled:", [f"{mn}/{cfg}" for mn, cfg in TRAINED_RUNS])

# separate spec/mfcc streams for the branched model (forward returns (spec, mfcc))
mel_train_plain_sep: SpectMFCCPipeline = SpectMFCCPipeline(combind=False).to(device)
mel_eval_sep: SpectMFCCPipeline = SpectMFCCPipeline(combind=False).to(device)
mel_eval_sep = torch.compile(mel_eval_sep) if device.type == "cuda" else mel_eval_sep

In [11]:
# ---- Multi-model 3-class trainer: model-by-string, best-checkpoint saves,
# early stop on val_loss plateaus, same crash-proof resume as train_3class ----
def _make_model(model_name: str, num_classes: int = 3) -> nn.Module:
    """Model factory by string. resnet = ResNet50 ImageNet weights, head swapped, full finetune."""
    if model_name == "default":
        return DefaultCNN(num_classes)
    if model_name == "branched":
        return BranchedCoughCNN(num_classes)
    if model_name == "resnet":
        return ResNetModel(num_classes, freeze_backbone=False)
    raise ValueError(f"unknown model_name {model_name!r} (default|branched|resnet)")

def _ckpt_best_path(name: str) -> str:
    return os.path.join(RUN_STATE_DIR, f"{name}_best.pth.tar")

def _resume_with_best(name: str, model: nn.Module, optimizer: optim.Optimizer,
                      scheduler: optim.lr_scheduler.LRScheduler,
                      scaler: torch.amp.GradScaler, num_epochs: int) -> tuple[int, dict, float, int]:
    """Load _latest checkpoint into model/opt/sched/scaler; derive best val_loss
    and trailing plateau length from history. Returns (epochs_done, history, best_vl, plateau)."""
    best_vl, plate = float("inf"), 0
    path = _ckpt_path(name)
    if not os.path.exists(path):
        return 0, {"train_loss": [], "train_acc": [], "train_F1": [], "val_loss": [], "val_acc": [], "val_F1": [], "val_auc": [], "val_pr_auc": []}, best_vl, plate
    ckpt = torch.load(path, weights_only=True, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    scaler.load_state_dict(ckpt["scaler_state_dict"])
    epochs_done: int = ckpt["epoch"] + 1  # 'epoch' is the 0-based index of the last COMPLETED epoch
    history: dict[str, list[float]] = ckpt["history"]
    for k in ("train_acc", "train_F1", "val_pr_auc"):  # old checkpoints predate these keys
        history.setdefault(k, []).extend([float("nan")] * (len(history["val_loss"]) - len(history.get(k, []))))
    if history["val_loss"]:
        best_vl = min(history["val_loss"])
        for vl in reversed(history["val_loss"]):
            if vl <= best_vl + 1e-6:
                break
            plate += 1
    if epochs_done >= num_epochs:
        trained_models_all[name] = model
    print(f"=> [{name}] resumed at epoch {epochs_done + 1}/{num_epochs} (best val_loss {best_vl:.4f})" if epochs_done < num_epochs
          else f"=> [{name}] already complete ({num_epochs}/{num_epochs} epochs) - restored, no retrain")
    return epochs_done, history, best_vl, plate

def _collect_probs_mn(model: nn.Module, mn: str, indices: Sequence[int]) -> tuple[np.ndarray, np.ndarray]:
    """Full (N, 3) softmax probs + labels for eval/curves; branched takes separate (spec, mfcc) streams."""
    loader = torch.utils.data.DataLoader(
        Subset(dataset, list(indices)), BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
    probs: list[list[float]] = []
    targets: list[int] = []
    model.eval()
    with torch.inference_mode():
        for waveforms, labels in loader:
            waveforms = waveforms.to(device, non_blocking=True)
            raw = labels.long()
            if mn == "branched":
                spec, mfcc = mel_eval_sep(waveforms)
                if spec.ndim == 3:
                    spec, mfcc = spec.unsqueeze(0), mfcc.unsqueeze(0)
                outputs = model(spec, mfcc)
            else:
                images = mel_eval(waveforms)
                if images.ndim == 3:
                    images = images.unsqueeze(0)
                outputs = model(images)
            probs.extend(torch.softmax(outputs.float(), dim=1).cpu().numpy())
            targets.extend(raw.numpy())
    return np.array(probs), np.array(targets)

def train_model_3class(name: str, use_wave: bool, use_spec: bool,
                       model_name: str = "default", num_epochs: int = TRAINED_EPOCHS,
                       patience: int = MODEL_PATIENCE) -> dict[str, list[float]]:
    """train_3class generalized: model by string, best-checkpoint saves, early stop
    when val_loss stalls for `patience` epochs. Crash-proof identical."""
    history: dict[str, list[float]] = {"train_loss": [], "train_acc": [], "train_F1": [], "val_loss": [], "val_acc": [], "val_F1": [], "val_auc": [], "val_pr_auc": []}

    w = 1.0 / torch.bincount(train_labels, minlength=3).float().clamp(min=1)
    w = w / w.sum() * 3.0  # normalize so mean weight is 1
    w = w.to(device)
    # Rarity-scaled augment probs from train-split class counts: rarer classes
    # get more augmentation (most-common -> 0.5, rarest -> 0.95).
    counts3 = torch.bincount(train_labels, minlength=3).float()
    aug_probs: dict[int, float] = aug_probs_from_counts({k: float(counts3[k]) for k in range(3)})
    print(f"[{name}] augment_probs: " + ", ".join(f"{k}:{v:.2f}" for k, v in sorted(aug_probs.items())))

    criterion: nn.Module = nn.CrossEntropyLoss(weight=w)  # same loss as train_3class: one definition across all 3-class runs

    m = _make_model(model_name)
    if device.type == "cuda":
        model: nn.Module = torch.compile(m).to(device)
    else:
        model = m.to(device)

    optimizer: optim.AdamW = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)
    scaler: torch.amp.GradScaler = torch.amp.GradScaler(enabled=(device.type == "cuda"))
    scheduler: optim.lr_scheduler.CosineAnnealingLR = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=num_epochs, eta_min=1e-6
    )

    epochs_done, history, best_val_loss, plateau = _resume_with_best(
        name, model, optimizer, scheduler, scaler, num_epochs)

    use_sep: bool = (model_name == "branched")
    if use_spec:
        # per-run pipeline: SpecAugmentations holds this run's augment_probs
        mel_train = SpectMFCCPipeline(
            transforms=SpecAugmentations(augment_probs=aug_probs).to(device),
            combind=not use_sep,
        ).to(device)
    else:
        mel_train = mel_train_plain_sep if use_sep else mel_train_plain

    train_ds: AugmentedSubset | Subset = AugmentedSubset(
        dataset, train_subset.indices, WaveformAugmentor(16_000, augment_probs=aug_probs)
    ) if use_wave else Subset(dataset, train_subset.indices)

    train_loader: torch.utils.data.DataLoader = torch.utils.data.DataLoader(
        train_ds, BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True,
        persistent_workers=True, prefetch_factor=4,
    )
    val_loader: torch.utils.data.DataLoader = torch.utils.data.DataLoader(
        Subset(dataset, val_subset.indices), BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True,
        persistent_workers=True, prefetch_factor=4,
    )

    def forward_features(waveforms: torch.Tensor, raw: torch.Tensor, mel):
        """Combined pipelines return one (B,3,F,T) image; separated return (spec, mfcc)."""
        feats = mel(waveforms, raw)
        if use_sep:
            spec, mfcc = feats
            if spec.ndim == 3:
                spec, mfcc = spec.unsqueeze(0), mfcc.unsqueeze(0)
            return spec, mfcc
        if feats.ndim == 3:
            feats = feats.unsqueeze(0)
        return feats

    for epoch in range(epochs_done, num_epochs):
        model.train()
        running_loss = 0.0
        train_preds: list[int] = []
        train_targets: list[int] = []

        for waveforms, labels in tqdm(train_loader, desc=f"[{name}] train {epoch+1}/{num_epochs}", leave=False):
            waveforms = waveforms.to(device, non_blocking=True)
            raw = labels.long().to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            feats = forward_features(waveforms, raw, mel_train)  # fp32 outside autocast (see train_3class note)
            with torch.amp.autocast(device_type=device.type, enabled=(device.type == "cuda")):
                outputs = model(*feats) if use_sep else model(feats)
                loss = criterion(outputs, raw)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item()
            train_preds.extend(torch.argmax(outputs, dim=1).cpu().numpy())
            train_targets.extend(raw.cpu().numpy())

        scheduler.step()
        train_acc = 100.0 * float(np.mean(np.asarray(train_targets) == np.asarray(train_preds)))
        train_f1 = f1_score(train_targets, train_preds, average="macro")

        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0
        all_preds, all_targets, all_probs3 = [], [], []

        with torch.inference_mode():
            for waveforms, labels in tqdm(val_loader, desc=f"[{name}] val {epoch+1}/{num_epochs}", leave=False):
                waveforms = waveforms.to(device, non_blocking=True)
                raw = labels.long().to(device, non_blocking=True)
                feats = forward_features(waveforms, raw, mel_eval_sep if use_sep else mel_eval)
                with torch.amp.autocast(device_type=device.type, enabled=(device.type == "cuda")):
                    outputs = model(*feats) if use_sep else model(feats)
                    loss = criterion(outputs, raw)
                all_preds.extend(torch.argmax(outputs, dim=1).cpu().numpy())
                all_targets.extend(raw.cpu().numpy())
                all_probs3.extend(torch.softmax(outputs.float(), dim=1).cpu().numpy())
                val_loss += loss.item()
                total += raw.size(0)
                correct += (torch.argmax(outputs, dim=1) == raw).sum().item()

        val_macro_f1 = f1_score(all_targets, all_preds, average="macro")
        val_auc_score = roc_auc_score(all_targets, all_probs3, multi_class="ovr", average="macro")
        val_pr_auc = average_precision_score(
            label_binarize(all_targets, classes=[0, 1, 2]), np.asarray(all_probs3), average="macro")
        train_epoch_loss = running_loss / len(train_loader)
        val_epoch_loss = val_loss / len(val_loader)
        val_accuracy = 100.0 * correct / total

        history["train_loss"].append(train_epoch_loss)
        history["val_loss"].append(val_epoch_loss)
        history["val_F1"].append(val_macro_f1)
        history["val_acc"].append(val_accuracy)
        history["val_auc"].append(val_auc_score)
        history["train_acc"].append(train_acc)
        history["train_F1"].append(train_f1)
        history["val_pr_auc"].append(val_pr_auc)

        state = {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict(),
            "history": history,
        }
        _atomic_torch_save(state, _ckpt_path(name))
        if val_epoch_loss < best_val_loss - 1e-6:
            best_val_loss = val_epoch_loss
            plateau = 0
            _atomic_torch_save(state, _ckpt_best_path(name))  # best checkpoint on val_loss improvement
            print(f"=> [{name}] new best val_loss {best_val_loss:.4f} (epoch {epoch+1})")
        else:
            plateau += 1

        print(f"[{name}] Epoch [{epoch+1}/{num_epochs}] -> "
              f"Train Loss: {train_epoch_loss:.4f} | Train Acc: {train_acc:.2f}% | Train F1: {train_f1:.4f} | "
              f"Val Loss: {val_epoch_loss:.4f} | "
              f"Val F1: {val_macro_f1:.4f} | Val AUC: {val_auc_score:.4f} | Val PR-AUC: {val_pr_auc:.4f} | "
              f"Val Acc: {val_accuracy:.2f}%")

        if plateau >= patience:
            print(f"=> [{name}] early stop: val_loss plateaued for {patience} epochs (best {best_val_loss:.4f})")
            break

    trained_models_all[name] = model
    return history

In [ ]:
# ---- Multi-model training loop: models x selected aug configs, 3-class ----
# All outputs labeled trained_*. Seeds from disk (never wipes/KeyErrors on re-run),
# resumes mid-run, short-circuits finished runs.
import time, sys, atexit
LOG_PATH3 = r"logs_3class/trained_models_experiment.txt"
_logf3 = open(LOG_PATH3, "a", encoding="utf-8")
sys.stdout = _logf3
sys.stderr = _logf3
atexit.register(_logf3.close)

TRAINED_CSV_COLS: list[str] = ["train_loss", "train_acc", "train_F1", "val_loss", "val_acc", "val_F1", "val_auc", "val_pr_auc"]

def seed_trained_state() -> tuple[dict, dict]:
    """Seed trained_results / trained_models_all from CSVs + checkpoints on disk
    (plus whatever is already in memory) so re-runs never wipe or KeyError."""
    results: dict[str, dict[str, list[float]]] = {}
    if "trained_results" in globals():
        results.update(trained_results)
    models_all: dict[str, nn.Module] = {}
    if "trained_models_all" in globals():
        models_all.update(trained_models_all)
    for mn, cfg in TRAINED_RUNS:
        key = f"{mn}__{cfg}"
        if key not in results and os.path.exists(f"trained_three_class_{key}_experiment_stats.csv"):
            df = pl.read_csv(f"trained_three_class_{key}_experiment_stats.csv")
            results[key] = {col: (df[col].to_list() if col in df.columns else [float("nan")] * df.height) for col in TRAINED_CSV_COLS}
        if key not in models_all and os.path.exists(_ckpt_path(key)):
            ckpt = torch.load(_ckpt_path(key), weights_only=True, map_location=device)
            m = _make_model(mn).to(device)
            with torch.inference_mode():  # materialize lazy params before load
                if mn == "branched":
                    s = torch.zeros(1, 3, 104, 32, device=device)
                    _ = m(s, s)
                else:
                    _ = m(torch.zeros(1, 3, 104, 32, device=device))
            if any(k.startswith("_orig_mod.") for k in ckpt["model_state_dict"]):
                m = torch.compile(m)
            m.load_state_dict(ckpt["model_state_dict"])
            m.eval()
            models_all[key] = m
    return results, models_all

trained_results, trained_models_all = seed_trained_state()
trained_stop_info: dict[str, str] = {}  # key -> "complete" | "early stop @ epoch N"

print("STARTING MULTI-MODEL 3-CLASS EXPERIMENTS")
for mn, cfg in TRAINED_RUNS:
    uw, us = AUG_MAP[cfg]
    key = f"{mn}__{cfg}"
    if key in trained_results and len(trained_results[key]["val_loss"]) >= TRAINED_EPOCHS:
        print(f"\n--- {key} --- already complete, skipping (restored from disk)")
        continue
    print(f"\n--- {key} ---")
    t0 = time.time()
    hist = train_model_3class(key, use_wave=uw, use_spec=us, model_name=mn, num_epochs=TRAINED_EPOCHS)
    trained_results[key] = hist
    pl.DataFrame({k: hist[k] for k in TRAINED_CSV_COLS}).write_csv(f"trained_three_class_{key}_experiment_stats.csv")
    done = len(hist["val_loss"])
    trained_stop_info[key] = "complete" if done >= TRAINED_EPOCHS else f"early stop @ epoch {done}"
    # final weights to saved_models_3class: best-checkpoint state as a plain .pth
    best = torch.load(_ckpt_best_path(key), weights_only=True, map_location=device)
    torch.save(best["model_state_dict"], f"saved_models_3class/trained_{key}.pth")
    print(f"=> {key}: {done}/{TRAINED_EPOCHS} epochs in {time.time() - t0:.0f}s - {trained_stop_info[key]}")

print("\n=== Multi-model 3-class training done ===")
for mn, cfg in TRAINED_RUNS:
    key = f"{mn}__{cfg}"
    if key in trained_results:
        h = trained_results[key]
        print(f"{key:<24} {len(h['val_loss']):>3}/{TRAINED_EPOCHS} epochs  final val_acc {h['val_acc'][-1]:.2f}%")

In [ ]:
# ---- Multi-model eval from checkpoints: per-class report + CMs on TEST ----
# Restores models from run_state_3class checkpoints (no training needed),
# runs the full eval, saves run_state_3class/trained_eval_state.pt + CSV.
import os
from sklearn.metrics import roc_curve, precision_score, recall_score

def _save_eval_state(obj: dict) -> None:
    tmp = "run_state_3class/trained_eval_state.pt.tmp"
    torch.save(obj, tmp)
    os.replace(tmp, "run_state_3class/trained_eval_state.pt")

def _load_eval_state() -> dict | None:
    p = "run_state_3class/trained_eval_state.pt"
    return torch.load(p, weights_only=False) if os.path.exists(p) else None

def _restore_trained_models() -> dict:
    """One model per TRAINED_RUNS key, straight from checkpoints.
    Checkpoints were saved from torch.compile'd models on CUDA -> keys may carry
    the '_orig_mod.' prefix; wrap in compile in that case."""
    models = {}
    for mn, cfg in TRAINED_RUNS:
        key = f"{mn}__{cfg}"
        path = _ckpt_path(key)
        assert os.path.exists(path), f"{path} missing - run the training cells first"
        ckpt = torch.load(path, weights_only=True, map_location=device)
        m = _make_model(mn).to(device)
        with torch.inference_mode():  # materialize lazy params before load
            if mn == "branched":
                s = torch.zeros(1, 3, 104, 32, device=device)
                _ = m(s, s)
            else:
                _ = m(torch.zeros(1, 3, 104, 32, device=device))
        if any(k.startswith("_orig_mod.") for k in ckpt["model_state_dict"]):
            m = torch.compile(m)
        m.load_state_dict(ckpt["model_state_dict"])
        m.eval()
        models[key] = m
    return models

if "trained_preds" in globals() and trained_preds:
    print("=> trained eval already in memory, skipping")
else:
    _ev = _load_eval_state()
    if _ev is not None:
        trained_preds = _ev["trained_preds"]; test_y3_t = _ev["test_y3"]
        trained_eval_rows = _ev["trained_eval_rows"]; trained_curves = _ev["curves"]
        trained_models_all = _restore_trained_models()
        print("=> trained eval state restored from run_state_3class/trained_eval_state.pt (eval pass skipped)")
    else:
        trained_models_all = _restore_trained_models()
        trained_preds: dict[str, np.ndarray] = {}
        trained_eval_rows: list[dict] = []
        trained_curves: dict[str, dict] = {}  # per run: per-class OvR ROC/PR curve data

        test_y3_t = np.array([int(dataset.labels[i]) for i in test_subset.indices])

        for mn, cfg in TRAINED_RUNS:
            key = f"{mn}__{cfg}"
            m = trained_models_all[key]
            probs3, y = _collect_probs_mn(m, mn, test_subset.indices)
            pred = probs3.argmax(axis=1)

            cm = confusion_matrix(y, pred, labels=[0, 1, 2])
            prec = precision_score(y, pred, labels=[0, 1, 2], average=None, zero_division=0)
            rec = recall_score(y, pred, labels=[0, 1, 2], average=None, zero_division=0)
            f1s = f1_score(y, pred, labels=[0, 1, 2], average=None, zero_division=0)
            trained_eval_rows.append({
                "experiment": f"trained_three_class_{key}",
                "healthy_recall": round(rec[0], 4), "covid_recall": round(rec[1], 4), "symp_recall": round(rec[2], 4),
                "healthy_precision": round(prec[0], 4), "covid_precision": round(prec[1], 4), "symp_precision": round(prec[2], 4),
                "healthy_f1": round(f1s[0], 4), "covid_f1": round(f1s[1], 4), "symp_f1": round(f1s[2], 4),
                "macro_f1": round(f1_score(y, pred, average="macro", zero_division=0), 4),
                "macro_roc_auc": round(roc_auc_score(y, probs3, multi_class="ovr", average="macro"), 4),
                "accuracy": round(float((pred == y).mean()), 4),
            })
            trained_preds[key] = pred
            print(f"\n=== TRAINED 3-class TEST [{key}] ===")
            print(f"(rows=true ['healthy','COVID-19','symptomatic'], cols=pred)\n{cm}")
            print(classification_report(y, pred, labels=[0, 1, 2],
                                        target_names=[LABELS_INV[0], LABELS_INV[1], LABELS_INV[2]], zero_division=0))

            # per-class one-vs-rest curve data for the plot cell (works on fresh kernel too)
            curves: dict[str, dict] = {}
            for c, cname in LABELS_INV.items():
                ybin = (y == c).astype(int)
                fpr, tpr, _ = roc_curve(ybin, probs3[:, c])
                pr, pr_t, _ = precision_recall_curve(ybin, probs3[:, c])
                curves[cname] = {
                    "roc_fpr": fpr, "roc_tpr": tpr, "pr_prec": pr, "pr_rec": pr_t,
                    "roc_auc": round(roc_auc_score(ybin, probs3[:, c]), 4),
                    "pr_auc": round(auc(pr_t, pr), 4),
                }
            trained_curves[key] = curves

        _save_eval_state({"trained_preds": trained_preds, "test_y3": test_y3_t,
                          "trained_eval_rows": trained_eval_rows, "curves": trained_curves})
        print("=> trained eval state saved to run_state_3class/trained_eval_state.pt")

pl.DataFrame(trained_eval_rows).write_csv("trained_three_class_eval_results.csv")
print("=> wrote trained_three_class_eval_results.csv")

In [ ]:
# ---- Multi-model plots + summary table (all into figures_3class/ + root CSVs) ----
# crash recovery: histories from CSVs, preds/curves from eval state
import matplotlib.pyplot as plt
if "trained_results" not in globals() or not trained_results:
    trained_results = {}
    for mn, cfg in TRAINED_RUNS:
        key = f"{mn}__{cfg}"
        csv = f"trained_three_class_{key}_experiment_stats.csv"
        if os.path.exists(csv):
            df = pl.read_csv(csv)
            trained_results[key] = {col: (df[col].to_list() if col in df.columns else [float("nan")] * df.height) for col in TRAINED_CSV_COLS}
    print("=> trained_results rebuilt from trained_three_class_*_experiment_stats.csv")

if "trained_preds" not in globals():
    _tep = "run_state_3class/trained_eval_state.pt"
    assert os.path.exists(_tep), "run_state_3class/trained_eval_state.pt missing - run the eval cell first"
    _tev = torch.load(_tep, weights_only=False)
    trained_preds, test_y3_t = _tev["trained_preds"], _tev["test_y3"]
    trained_eval_rows, trained_curves = _tev["trained_eval_rows"], _tev["curves"]
    print("=> trained preds/test_y3 restored from run_state_3class/trained_eval_state.pt")

# history plots: loss (train+val) / val acc / val F1 / val AUC
for (mn, cfg) in TRAINED_RUNS:
    key = f"{mn}__{cfg}"
    if key not in trained_results:
        continue
    hist = trained_results[key]
    fig, axes = plt.subplots(1, 5, figsize=(20, 4))
    fig.suptitle(f"trained {mn} / {cfg} (3-class)")
    epochs = range(1, len(hist["train_loss"]) + 1)
    axes[0].plot(epochs, hist["train_loss"], label="train")
    axes[0].plot(epochs, hist["val_loss"], label="val")
    axes[0].set_title("Loss"); axes[0].legend()
    axes[1].plot(epochs, hist["val_acc"]); axes[1].set_title("Val Accuracy (%)")
    axes[2].plot(epochs, hist["val_F1"]); axes[2].set_title("Val F1")
    axes[3].plot(epochs, hist["val_auc"]); axes[3].set_title("Val AUC (macro OvR)")
    axes[4].plot(epochs, hist["val_pr_auc"]); axes[4].set_title("Val PR-AUC (macro OvR)")
    fig.tight_layout()
    fig.savefig(f"figures_3class/trained_{key}_history.png", bbox_inches="tight")
    plt.show(fig)
    plt.close(fig)

# 3-class confusion matrices
for key, pred in trained_preds.items():
    cm = confusion_matrix(test_y3_t, pred, labels=[0, 1, 2])
    disp = ConfusionMatrixDisplay(cm, display_labels=list(LABELS_INV.values()))
    disp.plot(cmap="Blues")
    plt.title(f"Trained 3-class TEST - {key}")
    plt.savefig(f"figures_3class/trained_{key}_cm.png", bbox_inches="tight")
    plt.show()
    plt.close("all")

# per-class one-vs-rest ROC + PR curves per run
for key, cur in trained_curves.items():
    fig, (ax_r, ax_p) = plt.subplots(1, 2, figsize=(11, 4.5))
    fig.suptitle(f"trained {key} TEST (one-vs-rest)")
    for cname, c in cur.items():
        ax_r.plot(c["roc_fpr"], c["roc_tpr"], label=f"{cname} (AUC {c['roc_auc']})")
        ax_p.plot(c["pr_rec"], c["pr_prec"], label=f"{cname} (PR-AUC {c['pr_auc']})")
    ax_r.plot([0, 1], [0, 1], "k--", lw=0.8)
    ax_r.set_xlabel("FPR"); ax_r.set_ylabel("TPR"); ax_r.set_title("ROC"); ax_r.legend()
    ax_p.set_xlabel("Recall"); ax_p.set_ylabel("Precision"); ax_p.set_title("Precision-Recall"); ax_p.legend()
    fig.tight_layout()
    fig.savefig(f"figures_3class/trained_{key}_curves.png", bbox_inches="tight")
    plt.show(fig)
    plt.close(fig)

# comparison figure: COVID-19 (OvR) ROC, one line per run
fig, ax = plt.subplots(figsize=(6, 5))
for key, cur in trained_curves.items():
    c = cur["COVID-19"]
    ax.plot(c["roc_fpr"], c["roc_tpr"], label=f"{key} (AUC {c['roc_auc']})")
ax.plot([0, 1], [0, 1], "k--", lw=0.8)
ax.set_xlabel("FPR"); ax.set_ylabel("TPR"); ax.set_title("COVID-19 (OvR) ROC - TEST"); ax.legend()
fig.tight_layout()
fig.savefig("figures_3class/trained_test_curves_comparison.png", bbox_inches="tight")
plt.show(fig)
plt.close(fig)

# summary table + CSV (with epoch counts); eval CSV rewritten too so a
# fresh-kernel restore refreshes it
rows = []
for key, hist in trained_results.items():
    for metric in ("train_loss", "train_acc", "train_F1", "val_loss", "val_acc", "val_F1", "val_auc", "val_pr_auc"):
        vals = hist[metric]
        rows.append({
            "experiment": f"trained_three_class_{key}", "metric": metric,
            "max": round(max(vals), 4), "min": round(min(vals), 4),
            "final": round(vals[-1], 4), "avg": round(sum(vals) / len(vals), 4),
            "std": round(float(np.std(vals)), 4), "epochs": len(vals),
        })
summary_df = pl.DataFrame(rows)
pl.Config.set_ascii_tables()  # cp1252 console can't render polars box chars
with pl.Config(tbl_rows=-1, tbl_cols=-1, tbl_width_chars=120):
    print(summary_df)
summary_df.write_csv("trained_three_class_experiment_summary.csv")
pl.DataFrame(trained_eval_rows).write_csv("trained_three_class_eval_results.csv")
print("All trained-model figures + CSVs written to figures_3class/ and notebook root.")

In [12]:
# ---- 5-fold CV over recordings: same configs as the trained_* sweep ----
# Every train+val recording is validated exactly once across the 5 folds; the
# test split from the stratified split above stays untouched as the final
# holdout. Per-fold metric = value at that fold's best-val-loss epoch (same
# selection rule as the _best checkpoint). Epoch budget and patience are fixed
# ONCE here, identical across folds and configs, so fold results are comparable.
import sys, time, atexit

LOG_PATH_CV = r"logs_3class/cv_experiment.txt"
_logf_cv = open(LOG_PATH_CV, "a", encoding="utf-8")
sys.stdout = _logf_cv
sys.stderr = _logf_cv
atexit.register(_logf_cv.close)

CV_RUNS: list[tuple[str, str]] = TRAINED_RUNS  # mirrors the multi-model sweep; edit TRAINED_RUNS to sweep more
CV_EPOCHS: int = TRAINED_EPOCHS
CV_PATIENCE: int = MODEL_PATIENCE

cv_pool = list(train_subset.indices) + list(val_subset.indices)  # test recordings excluded
cv_paths = [dataset.paths[i] for i in cv_pool]
cv_labels = [int(dataset.labels[i]) for i in cv_pool]
cv_folds = stratified_recording_folds(cv_paths, cv_labels, n_splits=5, seed=SEED)

# runnable check: folds partition the pool, no recording validated twice
_va_recs = [{cv_paths[p] for p in va_pos} for _, va_pos in cv_folds]
assert sorted(p for _, va_pos in cv_folds for p in va_pos) == list(range(len(cv_pool))), "folds don't partition the pool"
assert sum(len(rs) for rs in _va_recs) == len(set(cv_paths)), "a recording landed in two folds"
for _, va_pos in cv_folds:  # every val fold stratified: all three classes present
    assert {cv_labels[p] for p in va_pos} == {0, 1, 2}, "val fold lost a class"

cv_rows: list[dict] = []
_orig = (train_subset, val_subset, train_labels)  # restore after CV: later cells expect the single split
try:
    for fi, (tr_pos, va_pos) in enumerate(cv_folds):
        train_subset = Subset(dataset, [cv_pool[p] for p in tr_pos])
        val_subset = Subset(dataset, [cv_pool[p] for p in va_pos])
        train_labels = torch.tensor([int(dataset.labels[i]) for i in train_subset.indices])  # refit weights per fold
        for mn, cfg in CV_RUNS:
            uw, us = AUG_MAP[cfg]
            key = f"cv__{mn}__{cfg}__f{fi}"
            print(f"\n=== fold {fi + 1}/{len(cv_folds)}  {mn}/{cfg} ===")
            t0 = time.time()
            hist = train_model_3class(key, use_wave=uw, use_spec=us, model_name=mn,
                                      num_epochs=CV_EPOCHS, patience=CV_PATIENCE)
            best_ep = int(np.argmin(hist["val_loss"]))
            cv_rows.append({
                "model": mn, "aug": cfg, "fold": fi, "epochs": len(hist["val_loss"]),
                "val_loss": round(hist["val_loss"][best_ep], 4),
                "val_acc": round(hist["val_acc"][best_ep], 4),
                "val_F1": round(hist["val_F1"][best_ep], 4),
                "val_auc": round(hist["val_auc"][best_ep], 4),
                "val_pr_auc": round(hist["val_pr_auc"][best_ep], 4),
            })
            print(f"=> {key}: best val_loss {hist['val_loss'][best_ep]:.4f} @ epoch {best_ep + 1} ({time.time() - t0:.0f}s)")
finally:
    train_subset, val_subset, train_labels = _orig

# aggregate: mean±std of best-epoch metrics across folds, per config
print("\n=== 5-fold CV summary (metrics at best-val-loss epoch per fold) ===")
summary_rows: list[dict] = []
for mn, cfg in CV_RUNS:
    rs = [r for r in cv_rows if r["model"] == mn and r["aug"] == cfg]
    if not rs:
        continue
    print(f"{mn}/{cfg}: epochs per fold {[r['epochs'] for r in rs]}")
    for metric in ("val_loss", "val_acc", "val_F1", "val_auc", "val_pr_auc"):
        summary_rows.append({
            "model": mn, "aug": cfg, "metric": metric,
            "mean": round(float(np.mean([r[metric] for r in rs])), 4),
            "std": round(float(np.std([r[metric] for r in rs])), 4),
            **{f"f{r['fold']}": r[metric] for r in rs},
        })
cv_summary = pl.DataFrame(summary_rows)
pl.Config.set_ascii_tables()  # cp1252 console can't render polars box chars
with pl.Config(tbl_rows=-1, tbl_cols=-1, tbl_width_chars=120):
    print(cv_summary)
cv_summary.write_csv("cv_three_class_summary.csv")
print("=> wrote cv_three_class_summary.csv")

[cv__default__both__f0] train 1/256:   0%|          | 0/51 [00:51<?, ?it/s]

[cv__default__both__f0] val 1/256:   0%|          | 0/13 [00:55<?, ?it/s]

[cv__default__both__f0] train 2/256:   0%|          | 0/51 [00:00<?, ?it/s]

[cv__default__both__f0] val 2/256:   0%|          | 0/13 [00:00<?, ?it/s]

[cv__default__both__f0] train 3/256:   0%|          | 0/51 [00:00<?, ?it/s]

[cv__default__both__f0] val 3/256:   0%|          | 0/13 [00:00<?, ?it/s]

... (19 outputs left out)

[cv__default__both__f0] val 13/256:   0%|          | 0/13 [00:00<?, ?it/s]

[cv__default__both__f0] train 14/256:   0%|          | 0/51 [00:00<?, ?it/s]

[cv__default__both__f0] val 14/256:   0%|          | 0/13 [00:00<?, ?it/s]

ValueError: Target scores need to be probabilities for multiclass roc_auc, i.e. they should sum up to 1.0 over classes